# CS336 Assignment 1 (basics): Building a Transformer LM


## Byte-Pair Encoding (BPE) Tokenizer


### The Unicode Standard


In [1]:
ord("牛")

29275

In [2]:
chr(29275)

'牛'

(a) What Unicode character does chr(0) return?


In [3]:
chr(0)

'\x00'

(b) How does this character’s string representation (**repr**()) differ from its printed
representation?

- character's string representation is the unambiguous, technical string
- printed representation is human readable, user-friendly output


In [4]:
print(chr(0))

 


(c) What happens when this character occurs in text?

- The null character is embedded invisibly in the string—it's present and counted in the length, but displays as nothing when printed.


In [5]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [6]:
print("this is a test" + chr(0) + "string")

this is a test string


### Unicode Encodings


In [7]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'


In [8]:
print(type(utf8_encoded))

<class 'bytes'>


In [9]:
# Get the byte values for the encoded string (integers from 0 to 255).
list(utf8_encoded)

[104,
 101,
 108,
 108,
 111,
 33,
 32,
 227,
 129,
 147,
 227,
 130,
 147,
 227,
 129,
 171,
 227,
 129,
 161,
 227,
 129,
 175,
 33]

In [10]:
# One byte does not necessarily correspond to one Unicode character!
print(len(test_string))

13


In [11]:
print(len(utf8_encoded))

23


In [12]:
print(utf8_encoded.decode("utf-8"))

hello! こんにちは!


(a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
UTF-16 or UTF-32?

- UTF-8 is preferred because it is the most compact encoding for ASCII-heavy text (1 byte per character vs. 2 or 4), resulting in a smaller base vocabulary (256 byte values vs. 65,536+ for UTF-16/UTF-32), which makes tokenizer training more efficient.


(b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string
into a Unicode string. Why is this function incorrect? Provide an example of an input byte
string that yields incorrect results.

- It fails because the function decodes each byte individually, but multi-byte UTF-8 characters (like é, which is 2 bytes) cannot be decoded as separate single bytes.


In [13]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])


decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [14]:
decode_utf8_bytes_to_str_wrong("é".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data

(c) Give a two-byte sequence that does not decode to any Unicode character(s).

- b'\xc0\x00' is invalid UTF-8 because 0xc0 signals a two-byte sequence but 0x00 is not a valid continuation byte (continuation bytes must be in the range 0x80–0xBF).


### Subword Tokenization


### BPE Tokenization Training


In [15]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [16]:
import regex as re

re.findall(PAT, "some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [17]:
from cs336_basics.tokenizer import train_bpe

import os

path = os.path.join("..", "tests/fixtures", "tinystories_sample.txt")

print(path)

vocab, merges = train_bpe(path, vocab_size=266, verbose=True)

../tests/fixtures/tinystories_sample.txt
Initialized vocab with 256 tokens (including special tokens).
Training BPE with target vocab size 266 (10 merges to perform).
Reading input file: ../tests/fixtures/tinystories_sample.txt
Merge 1: (b' ', b't') (freq: 97)
Merge 2: (b'h', b'e') (freq: 78)
Merge 3: (b' ', b'a') (freq: 72)
Merge 4: (b' ', b's') (freq: 61)
Merge 5: (b'n', b'd') (freq: 50)
Merge 6: (b' ', b'w') (freq: 47)
Merge 7: (b'e', b'd') (freq: 42)
Merge 8: (b' ', b'f') (freq: 42)
Merge 9: (b' ', b'h') (freq: 38)
Merge 10: (b'i', b't') (freq: 36)


In [18]:
import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()

path = os.path.join("..", "data", "TinyStoriesV2-GPT4-train.txt")
vocab, merges = train_bpe(path, vocab_size=10000, special_tokens=["<|endoftext|>"])

profiler.disable()
stats = pstats.Stats(profiler)
stats.sort_stats("cumulative")  # or 'tottime' for self-time only
stats.print_stats(20)  # top 20 functions


FileNotFoundError: Input file not found: ../data/TinyStoriesV2-GPT4-train.txt

In [ ]:
stats.sort_stats("tottime")  # or 'cumulative' for cumulative time
stats.print_stats()

         759913933 function calls (747474532 primitive calls) in 577.040 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
10870808/10870802   67.940    0.000   75.317    0.000 {built-in method posix.read}
  5435403   54.023    0.000   83.801    0.000 {built-in method _pickle.loads}
11426360/11413725   52.968    0.000  105.602    0.000 /Users/yashashwitashukla/.local/share/uv/python/cpython-3.12.11-macos-aarch64-none/lib/python3.12/collections/__init__.py:669(update)
      133   48.501    0.365   56.963    0.428 {method 'control' of 'select.kqueue' objects}
255484114   42.365    0.000   42.365    0.000 {method 'get' of 'dict' objects}
  5435430   29.759    0.000   29.760    0.000 {method 'dump' of '_pickle.Pickler' objects}
  1805452   23.683    0.000   23.752    0.000 {built-in method posix.waitpid}
  5435399   20.645    0.000  478.330    0.000 /Users/yashashwitashukla/.local/share/uv/python/cpython-3.12.11-macos-aarch64-non